# Clase 7
## Limpieza de datos

In [ ]:
import seaborn as sns

import seaborn.objects as so
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import sqlite3

En este trabajo exploraremos la base de datos `Sleep_Efficiency_Cleaning.csv`. Es una base descargada de kaggle.com generada a partir de información sobre los hábitos de sueño de distintas personas. La base está levemente modificada para los ejercicios de este trabajo. Las variables incluidas son:

- **ID:** identificación de participante
- **Age:** edad
- **Gender:** que designa al género de cada persona
- **Bedtime:** que tiene la hora a la que se va a dormir la persona
- **Wakeup time:** que tiene la hora a la que se despiertan las personas
- **Sleep.duration:**  que nos dice cuánto tiempo pasa cada persona durmiendo
- **Sleep.efficiency:** que nos dice cuanto del tiempo que pasan laa persona en la cama, efectivamente duermen.
- **REM sleep percentage:**  que tiene la proporción de sueño REM de la persona
- **Deep sleep percentage:**  que tiene la proporción de sueño profundo de la persona
- **Light sleep percentage:**  que tiene la proporción de sueño liviano de la persona
- **Awakenings:** que tiene la cantidad de veces que la persona se despierta a la noche
- **Caffeine consumption:** que nos dice la cantidad de cafeina que consume la persona
- **Alcohol consumption:** que nos dice la cantidad de alcohol que consume la persona
- **Smoking.status:** que nos dice si la persona es o no es fumadora
- **Excersice frequency:** que nos dice cuán frecuentemente hace ejercicio la persona

In [ ]:
datos = pd.read_csv("Sleep_efficiency_cleaning.csv")
datos   

#### 1. Con lo que sabemos hasta ahora, ¿pueden detectar cosas para limpiar en los datos?

### 2. Limpieza de los datos

Nos interesará realizar algunas acciones de limpieza de datos:
- Filtrar datos
- Renombrar variables
- Eliminar datos faltantes
- Eliminar duplicados

#### 2.a. Filtrado

Veamos los histogramas de edades y duración del sueño.


In [ ]:
(
    so.Plot(datos, "Sleep duration")
    .add(so.Bars(), so.Hist(bins = 10))
)

In [ ]:
(
    so.Plot(datos, "Age")
    .add(so.Bars(), so.Hist())
)

En base a estos histogramas, podemos eliminar datos con valores poco frecuentes.

¿Cómo podemos sacar de los datos a los participantes menores de edad y a aquellos que hayan dormido menos de 7 horas?


In [ ]:
# Filtramos datos usando SQL
con = sqlite3.connect(":memory:")   # Le damos otro nombre a la conexión, representa una conexión a otra base de datos.
datos.to_sql("sleep", con, index=False, if_exists="replace")

datos = pd.read_csv("Sleep_efficiency_cleaning.csv")

datos = pd.read_sql_query("???", con)
datos   

In [ ]:
# Para nombres con espacios encerramos el nombre entre comillas dobles, dentro de un string las ingresamos con \"
datos = pd.read_sql_query("???", con)
datos

### 2.b. Renombrar variables con diccionarios

Como vimos, nuestra base de datos tiene nombres de variables en inglés. Pero como nosotros somos más argentinos que usar ojotas con medias, vamos a pasarlo al castellano.

Para eso, nos van a servir mucho los **diccionarios**, que se utilizan para almacenar valores de datos en pares clave:valor.

Un diccionario es una colección ordenada, modificable y que no permite duplicados.

Yo podría generar este diccionario llamado dict

In [ ]:
# Ejemplo
dict = { "hola" : "Expresión de saludo utilizada entre dos o más personas de trato familiar", "cinco" : 5, "dias" : ["lunes", "martes", "miercoles"]}

Si le pregunto que significa "hola", me dirá:

In [ ]:
dict["hola"]

In [ ]:
# Ejemplo
thisdict = {
  "brand": "Ford",
  "model": "Mustang",
  "year": 1964
}
print(thisdict["brand"])

Para lo que nos interesa a nosotros, podríamos tener lo siguiente:

In [ ]:
nombres = {
  "ID": "ID",
  "Age": "Edad",
  "Gender": "Género",
  "Bedtime": "Hora dormir",
  "Wakeup time": "Hora despertar",
  "Sleep duration": "Duracion suenio",
  "Sleep efficiency": "Eficiencia suenio",
  "REM sleep percentage": "Porcentaje suenio REM",
  "Deep sleep percentage": "Porcentaje suenio profundo",
  "Light sleep percentage": "Porcentaje suenio liviano",
  "Awakenings": "Despertares",
  "Caffeine consumption": "Consumo cafeina",
  "Alcohol consumption": "Consumo alcohol",
  "Smoking status": "Fumador",
  "Exercise frequency": "Frecuencia ejercicio"
}

Para cambiar los nombres deberíamos usar rename() ¿Cómo lo usarian?

Pueden chusmear la ayuda

In [ ]:
help(pd.DataFrame.rename)

#### Syntaxis: column = function

In [ ]:
datos.rename(columns = nombres)

In [ ]:
# Funcionó?
datos

In [ ]:
# Tenemos que asignarlo para que se guarden los cambios
datos = datos.rename(columns= nombres)
datos

**Pregunta:** Podemos hacer esto directamente en SQL? Sí, pero necesitamos crear la instrucción apropiada usando Python. Vamos a limitar el uso de SQL a filtrar o agregar datos.

#### Syntaxis: column = function

También podemos usar funciones para renombrar columnas.

**Ejemplo:** Pasar todos los nombres de columnas a mayúsculas.

In [ ]:
# Para pasar un string a mayusculas utilizamos str.upper
str.upper("hola")


In [ ]:
# También
str.lower("Cómo te va?")


¿Cómo lo usarian para renombrar nuestras variables?

In [ ]:
# Así?
#datos.rename(columns = str.upper)
# o así?
#datos.rename(columns = str.upper())

Le pasamos el nombre de la función, por eso va sin los paréntesis. 
Si la pasamos con paréntesis le estaríamos pasando el resultado de evaluar la función sin parámetros.

#### 2. c. Eliminar valores faltantes

¿Qué significa un valor faltante? Es un dato que por distintos motivos no fue ingresado (por ejemplo si juntamos datos de distintas fuentes, en alguna de las fuentes esa información puede no estar disponible).

Los datos faltantes son algo típico de cualquier pipeline de análisis de datos. En general los paquetes estadísticos (pandas, dplyr, etc) tratan de facilitar el manejo de estos tipos de datos. En Pandas, se utiliza la expresión NaN (Not a number) o NA que viene del mundo R (Not available)

Para poder acceder a los datos faltantes, podemos utilizar la función isna() 

¿Cómo la usamos? ¿Qué nos devuelve? 

In [ ]:
datos.isna()

¿Cómo podemos transformar toda esta información para que sea más legible?

In [ ]:
# Una forma bien simple sería usando 
datos.isna().sum()

# De ahi sabemos que Despertares, Consumo de cafeina, Consumo de alcohol y Frecuencia de ejercicio tienen muchos datos faltantes

In [ ]:
# Esto nos da lo mismo (null es comun en SQL, NaN se usa en NumPy)
datos.isnull().sum()

¿Qué podríamos hacer con todos estos datos faltantes?

In [ ]:
# Una posibilidad es usar dropna() 
# ¿Qué les parece que hace esta función? 
# ¿Cuántas filas teniamos antes y cuántas tenemos ahora? ¿Cuáles son las ventajas y desventajas de esto?

datos.dropna()

In [ ]:
# Otra posibilidad es eliminar las columnas con datos faltantes. De forma de no perder participantes. ¿Cuáles son las ventajas y desventajas de esto?
datos.dropna(axis="columns")

Otra opción es hacer una combinación de estas dos posibilidades. Por ejemplo, eliminar las columnas con muchos datos faltantes (e.g. Awakenings y Caffeine compsumption) y recien luego de eso eliminar las filas que tengan algun dato perdido. 

De esa forma no pierdo tantas filas ni tantas columnas.

In [ ]:
datos1 = datos.drop(columns=["Despertares", "Consumo cafeina"])
datos2 = datos1.dropna()
datos2

# En un rato vamos a ver que hay formas más eficientes y ordenadas de encadenar comandos como estos.

También podríamos elegir no eliminar los casos, sino reemplazar los datos faltantes con un valor (e.g. cero). A esto se lo llama "imputación de datos faltantes".

Lo podemos hacer con fillna()

In [ ]:
datos.fillna(0)

¿Qué opinamos de esto? ¿Se puede usar en todos los casos? ¿Qué otras formas se te ocurren?

In [ ]:
## Una forma super sencilla (pero no infalible) es reemplazar el dato perdido por la media o la mediana poblacional (Capaz la mediana es mejor porque te va a buscar un dato existente)

datos.fillna(datos.mean)

En pocas palabras:
- Completamos con 0 si pensamos que no figura el dato porque no hay o no corresponde. Por ejemplo, en información de alimentos y bebidas, el porcentaje de alcohol puede figurar como NA.
- Completamos con el promedio si pensamos que se perdió el dato y queremos afectar lo menos posible los modelos.

Cómo completarías estos datos faltantes?
- Metros cuadrados de un departamento.
- Cantidad de votos recibidos por un partido en la elección anterior.
- Cantidad de ventas realizadas de cierto producto en cierto día.
- Altura de un paciente.

### Ejercicio
1. Examinar el dataset de eficiencia MPG y hacer algo con los datos faltantes. ¿Qué harían si no queremos eliminar filas?
2. Examinar el dataset de pingüinos y hacer algo con los datos faltantes.

In [ ]:
penguins = sns.load_dataset("penguins")
mpg = sns.load_dataset("mpg")

### 2.d. Eliminar datos duplicados

En muchos casos puede suceder que haya datos duplicados en nuestro dataset. ¿Se les ocurren ejemplos?

Para identificarlos, podemos usar duplicated(). ¿Qué está haciendo esta función?

In [ ]:
datos.duplicated()

In [ ]:
# Cuántos datos duplicados hay?
datos.duplicated().sum()

Para eliminarlos, podríamos usar  drop_duplicates()


In [ ]:
datos.drop_duplicates()

¿Qué hace esta función? ¿Que valores se queda y cuales elimina?

Por defecto, se queda con el primer caso, pero podemos indicarle que se quede con el último por ej:

In [ ]:
datos.drop_duplicates(keep = "last")

**Pregunta:** ¿Podemos hacerlo directo en SQL?

Si tuvieramos sujetos duplicados. Es decir, completaron dos veces la misma encuesta. como lo podríamos saber?

In [ ]:
datos["ID"].duplicated().sum()

¿Y cómo los podríamos eliminar?

In [ ]:
datos.drop_duplicates(subset = "ID", keep= "last")

### 2.e. Resetear el index

Una vez que eliminamos filas por el motivo que sea, nos van a quedar medio raros index. Veamos como nos quedó datos2

In [ ]:
datos = pd.read_csv("Sleep_efficiency_cleaning.csv")
con = sqlite3.connect(":memory:")   # Le damos otro nombre a la conexión, representa una conexión a otra base de datos.
datos.to_sql("sleep", con, index=False, if_exists="replace")

datos = pd.read_sql_query("""
SELECT * 
FROM sleep
WHERE "Sleep duration" >= 7 and Age >= 18
""", con)

In [ ]:
nombres = {
  "ID": "ID",
  "Age": "Edad",
  "Gender": "Género",
  "Bedtime": "Hora dormir",
  "Wakeup time": "Hora despertar",
  "Sleep duration": "Duracion suenio",
  "Sleep efficiency": "Eficiencia suenio",
  "REM sleep percentage": "Porcentaje suenio REM",
  "Deep sleep percentage": "Porcentaje suenio profundo",
  "Light sleep percentage": "Porcentaje suenio liviano",
  "Awakenings": "Despertares",
  "Caffeine consumption": "Consumo cafeina",
  "Alcohol consumption": "Consumo alcohol",
  "Smoking status": "Fumador",
  "Exercise frequency": "Frecuencia ejercicio"
}
datos1 = datos.rename(columns= nombres)

In [ ]:
datos2 = datos1.drop(columns=["Despertares", "Consumo cafeina"])
datos3 = datos2.dropna()
datos3

In [ ]:
# Hay 387 filas pero el índice llega hasta 406.

In [ ]:
# Va a resetear los índices y crear una variable con los índices viejos
datos3.reset_index()

In [ ]:
# De esta forma solo nos quedamos con lo síndices nuevos.
datos3.reset_index(drop=True)

### 3. Encadenar métodos

Esta forma de ir modificando un dataset y guardandolo cada vez en otro dataset (o en el mismo) no es muy práctica.

Por ejemplo si ejecutamos varias veces el mismo comando sin renombrar los datos, obtendremos cada vez algo distinto.

Si estamos creando un dataframe nuevo cada vez que hacemos algo, vamos a tener la memoria llena de objetos prácticamente iguales

Una solución es encadenar métodos para hacer el código más legible y fácil de reusar. Y más eficiente también.

Cada función genera un nuevo dataframe. Asi que siempre tengo que usar funciones que me generen un dataframe.

In [ ]:
datos.rename(columns= nombres).drop(columns=["Despertares", "Consumo cafeina"])

¿Cómo incorporarian todo lo otro que fuimos haciendo? 
1. Traducir los nombres de columnas a español
2. Eliminar las columnas "Despertares" y "Consumo cafeina"
3. Eliminar las filas con datos faltantes que hayan quedado
4. Eliminar filas repetidas
5. Resetear los índices

### Funciones lambda


**Ejemplo:** si tenemos índices de 0 a N-1, ¿cómo podemos renumerarlos de 1 a N?

In [ ]:
# Trabajamos con los pingüinos
penguins = sns.load_dataset("penguins").dropna().reset_index(drop = True)
penguins

In [ ]:
# Una posibilidad es pasarle un nuevo vector de índices
penguinsCopy = penguins.copy()
penguinsCopy.index = penguinsCopy.index+1
penguinsCopy

Pero esto no devuelve nada, no lo podemos encadenar con otras transformaciones.
Para encadenarlo, podemos usar rename, que funciona también para índices.

In [ ]:
# Por ejemplo, tomamos raíz cuadrada de todos los índices
penguins.rename(index = np.sqrt)

Para sumar uno a todos los índices no podemos hacer penguinsClean.rename(index = + 1).

Podemos definir primero nuestra propia función.

In [ ]:
???

penguins.rename(index = sumaUno)

Resulta engorroso definir una función para usarla solamente dentro del rename. 

Las **funciones lambda** nos permiten definir funciones anónimas "al vuelo".

La sintaxis es

`lambda arguments : expression`

In [ ]:
# Ejemplo
x = lambda a : a + 10
print(x(5))

In [ ]:
# Ejemplo
x = lambda a, b : a * b
print(x(5, 6))

In [ ]:
# Ejemplo
x = lambda a : a + 1
print(x(5))

In [ ]:
# Ahora sí!
penguins.rename(index = lambda a : a + 1)

Esta operación la podemos encadenar con las anteriores:

In [ ]:
penguinsClean = penguins.dropna().reset_index(drop = True).rename(index = lambda a : a + 1)
penguinsClean.head()

In [ ]:
# O si queremos cada metodo en una linea nueva
penguinsClean = (
    penguins.dropna()
    .reset_index(drop = True)
    .rename(index = lambda a : a + 1)
)
penguinsClean.head()

### Ejercicio
Utilizando funciones lambda:
1. Encadenar un método más para agregar "_orig" a los nombres de todas las columnas.
2. Encadenar un método más para reemplazar todos _ por - en los nombres de columnas.

In [ ]:
# Sugerencia 1
"hola" + "chau"

In [ ]:
# Sugerencia 2
str.replace("abcndjaA", "a", "oo")